# 01 · 데이터셋 준비

원본 CSV 어노테이션과 이미지를 YOLO 형식으로 변환합니다.  
실행 전 `WORKSPACE_ROOT/data/` 아래에 다음 파일이 있어야 합니다:
- `labels_with_split.csv`
- `dataset 2/` (이미지 폴더)

## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
ANNOTATIONS_CSV = Path(globals().get('ANNOTATIONS_CSV',
                        str(WORKSPACE_ROOT / 'data' / 'labels_with_split.csv')))
IMAGES_DIR      = Path(globals().get('IMAGES_DIR',
                        str(WORKSPACE_ROOT / 'data' / 'dataset 2')))
OUTPUT_DIR      = Path(globals().get('OUTPUT_DIR',
                        str(WORKSPACE_ROOT / 'data' / 'processed' / 'yolo_dataset')))
FORCE_REBUILD   = bool(globals().get('FORCE_REBUILD', False))
SMOKE_TEST      = bool(globals().get('SMOKE_TEST', False))   # True: split당 32장만 빌드
SEED            = int(globals().get('SEED', 42))
# ──────────────────────────────────────────────────────────────────

print(f'ANNOTATIONS_CSV : {ANNOTATIONS_CSV}')
print(f'IMAGES_DIR      : {IMAGES_DIR}')
print(f'OUTPUT_DIR      : {OUTPUT_DIR}')
print(f'FORCE_REBUILD   : {FORCE_REBUILD}')
print(f'SMOKE_TEST      : {SMOKE_TEST}')

## ② 환경 초기화

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)

from mad.colab_utils import setup_colab_env
setup_colab_env(REPO_DIR, WORKSPACE_ROOT)

# 입력 파일 존재 확인
for path, label in [(ANNOTATIONS_CSV, 'labels_with_split.csv'), (IMAGES_DIR, 'images dir')]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {label}: {path}')

## ③ YOLO 데이터셋 빌드

In [ ]:
from mad.dataset_builder import DatasetBuildConfig, build_yolo_dataset

result = build_yolo_dataset(
    DatasetBuildConfig(
        annotations_csv=ANNOTATIONS_CSV,
        images_dir=IMAGES_DIR,
        output_dir=OUTPUT_DIR,
        force=FORCE_REBUILD,
        symlink=False,
        workspace_root=WORKSPACE_ROOT,
        max_images_per_split=32 if SMOKE_TEST else None,
        shuffle=SMOKE_TEST,
        seed=SEED,
        validate=True,
    )
)

print(f"\n✅ 데이터셋 빌드 완료")
print(f"   dataset_yaml      : {result['dataset_yaml']}")
print(f"   class_count       : {result['class_count']}")
print(f"   processed_images  : {result['stats']['processed_images']}")
if result.get('validation'):
    v = result['validation']
    print(f"   validation_valid  : {v['valid']}")
    for sp, info in v.get('split_summary', {}).items():
        print(f"   [{sp}] images={info.get('image_count',0)}, missing_labels={info.get('missing_labels',0)}")

## ④ 빌드 요약 확인

In [ ]:
import json
from pathlib import Path

output_dir = Path(result['output_dir'])
for fname in ['build_summary.json', 'validation_summary.json']:
    fpath = output_dir / fname
    if fpath.exists():
        print(f'--- {fname} ---')
        data = json.loads(fpath.read_text(encoding='utf-8'))
        import pprint; pprint.pprint(data)
        print()

## 다음 단계

`notebooks/03_run_benchmark.ipynb` 에서 벤치마크 실험을 실행하세요.

생성된 `dataset.yaml` 경로:
```
{OUTPUT_DIR}/dataset.yaml
```